In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 20.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00:00:0100:01


In [2]:
from datasets import load_dataset
import kagglehub
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
import torch
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [3]:
path = kagglehub.dataset_download("bitext/bitext-gen-ai-chatbot-customer-support-dataset")
print(path)

100%|██████████| 2.87M/2.87M [00:00<00:00, 188MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/bitext/bitext-gen-ai-chatbot-customer-support-dataset/versions/1


In [4]:
print(os.listdir(path))

['Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv']


In [5]:
dataset = load_dataset(
    "csv",
    data_files=f"{path}/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [6]:
print(dataset["train"][0])

{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [7]:
def format_example(example):
  return{
      "text":f"""Instruction:
  {example['instruction']}

  Response:
  {example['response']}"""
  }

In [8]:
formatted_dataset = dataset["train"].map(format_example)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

Instruction:
  question about cancelling order {{Order Number}}

  Response:
  I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [9]:
formatted_dataset = formatted_dataset.select(range(1000))

In [22]:
len(formatted_dataset)

1000

In [10]:
print(formatted_dataset[10]["text"])

Instruction:
  i dont know what to do to cancel order {{Order Number}}

  Response:
  I've taken note that you're unsure of the cancellation process for your order with the order number {{Order Number}}. Don't worry, I'm here to help you! To cancel your order, please follow these steps:

1. Sign into Your Account: Access our platform by signing in to your {{Online Company Portal Info}}. 
2. Find Your Order: Once logged in, navigate to your '{{Online Order Interaction}}' or '{{Online Order Interaction}}' tab to locate the order with the order number {{Order Number}}.
3. Initiate Cancellation: Click on the order and look for the option labeled '{{Online Order Interaction}}'. Select this option to initiate the cancellation process.
4. Confirm Cancellation: The system might prompt you to confirm the cancellation. If so, kindly provide the necessary information to proceed.
5. Review Cancellation Details: After confirming, you will receive a cancellation confirmation with the relevant detail

In [11]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [12]:
sample = formatted_dataset[0]["text"]

tokens = tokenizer(sample)

print(tokens.keys())
print(len(tokens["input_ids"]))

KeysView({'input_ids': [1, 3901, 3880, 29515, 781, 29473, 3764, 1452, 1309, 29485, 4340, 2513, 14717, 6790, 9746, 1743, 781, 781, 29473, 12875, 29515, 781, 29473, 1083, 29510, 1101, 9756, 1136, 1274, 1032, 3764, 8985, 12119, 1056, 2513, 14717, 6790, 9746, 11549, 1072, 1083, 29510, 29487, 2004, 1066, 3852, 1136, 1163, 1040, 2639, 1136, 1695, 29491, 6687, 1344, 7048, 1072, 2228, 1342, 3764, 29493, 1072, 1083, 29510, 1352, 1279, 1354, 2257, 1066, 6799, 1136, 29491], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})
72


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.float16
)

model  = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto"
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

In [ ]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [15]:
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,254,839,296 || trainable%: 0.0939


In [19]:
training_args = TrainingArguments(
    output_dir = "./crm-support-model",
    num_train_epochs = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate = 2e-4,
    logging_steps = 10,
    save_strategy = "steps",
    save_steps = 100
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset = formatted_dataset,
    args = training_args,
    max_seq_length = 256
)
model.gradient_checkpointing_enable()

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model("/crm-support-model")
tokenizer.save_pretrained("./crm-support-model")

In [ ]:
!zip -r crm-support-model.zip crm-support-model

In [ ]:
from google.colab import files

files.download("crm-support-model.zip")